In [ ]:
# # 02 - Difference-in-Differences & Price Elasticity
# Model: `ln Q = a + b*Treat + g*Post + d*(Treat x Post)` with HC3 robust errors.
# `d` is the causal log change in orders from the 5% discount. Elasticity = `d / ln(0.95)`.
# Treated vs controls: Region 3 (food) and Region 4 (delivery fee) against Regions 1-2.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.data import ROOT, load_orders, did_sample
from src import models

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})
FIG = ROOT / "figures"
LEVERS = [(3, "food"), (4, "fee")]
df = load_orders()

In [ ]:
# ## Main estimates

res, rows = {}, []
for region, lever in LEVERS:
    s = did_sample(df, region)
    d, p = models.did_log(s), models.did_poisson(s)
    res[lever] = dict(sample=s, did=d, pois=p)
    lo, hi = d["elasticity_ci"]
    rows.append({
        "region": region, "lever": lever,
        "effect on orders": f"{np.expm1(d['delta']):+.0%}",
        "delta (SE)": f"{d['delta']:.3f} ({d['delta_se']:.3f})",
        "elasticity": round(d["elasticity"], 2),
        "95% CI": f"[{lo:.1f}, {hi:.1f}]",
        "Poisson elasticity": round(p["elasticity"], 2),
    })
pd.DataFrame(rows).set_index("region")


In [ ]:
# Full regression output for the log model (Region 3):

print(res["food"]["did"]["model"].summary().tables[1])

# ## Is the comparison credible?
# 1. **Pre-trend test:** do treated and control regions trend differently before the policy?
# 2. **Placebo test:** a fake policy date inside the pre-period should show no effect.
# 3. **Event study:** weekly gaps should be flat before 1 March.

diag = []
for region, lever in LEVERS:
    s = res[lever]["sample"]
    pre, plc = models.pretrend_test(s), models.placebo_did(s)
    diag.append({"region": region, "pre-trend slope diff": round(pre["coef"], 3),
                 "pre-trend p": round(pre["p"], 3),
                 "placebo DiD": round(plc["coef"], 3), "placebo p": round(plc["p"], 3)})
pd.DataFrame(diag).set_index("region")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (r, lever) in zip(axes, LEVERS):
    es = models.event_study(res[lever]["sample"])
    ax.errorbar(es.week, es.coef, yerr=1.96 * es.se, fmt="o", capsize=3)
    ax.axhline(0, color="grey", lw=.8); ax.axvline(-.5, ls=":", color="k")
    ax.set_title(f"Event study: Region {r} ({lever} discount)")
    ax.set_xlabel("Weeks from 1 Mar 2023"); ax.set_ylabel("Treated-control gap in ln(orders)")
plt.tight_layout(); plt.savefig(FIG / "event_study.png", dpi=150); plt.show()

In [ ]:
# ## Interpretation
# - All diagnostics pass (p > 0.05), so the design supports the *direction* of the effect.
# - The implied elasticities (about -9 to -19) are far larger than anything plausible for
#   a 5% price change. Treated regions rose 65-100% in both cases, even though the fee
#   discount is only about $0.38 on a $23.50 total price. This most likely reflects how the
#   synthetic data was generated, so treat the magnitudes with caution.
# - Both estimates have wide confidence intervals; the optimisation notebook carries
#   them through.
#
# A model that includes `log(delivery fee)` and `log(distance)` together is perfectly
# collinear, because fee = $0.50 x distance. The DiD specification above avoids this.